In [1]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"D:\ME AND MY LIFE\InfraScan")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# SDNET2018 folder -> surface, label
FOLDER_LABELS = {
    ("D", "CD"): ("Deck", "Cracked"),
    ("D", "UD"): ("Deck", "Uncracked"),
    ("P", "CP"): ("Pavement", "Cracked"),
    ("P", "UP"): ("Pavement", "Uncracked"),
    ("W", "CW"): ("Wall", "Cracked"),
    ("W", "UW"): ("Wall", "Uncracked"),
}

# Reload the manifest so this notebook works independently.
manifest_path = PROCESSED_DIR / "image_manifest.csv"
df = pd.read_csv(manifest_path)

def get_folder_labels(path):
    relative = Path(path).relative_to(RAW_DIR)
    parts = relative.parts

    if len(parts) < 3:
        return ("Unknown", "Unknown")

    return FOLDER_LABELS.get(
        (parts[0].upper(), parts[1].upper()),
        ("Unknown", "Unknown")
    )

df[["surface_type", "label"]] = pd.DataFrame(
    df["path"].apply(get_folder_labels).tolist(),
    index=df.index
)

usable_df = df[
    df["is_valid"].astype(bool)
    & df["label"].isin(["Cracked", "Uncracked"])
].copy()

usable_df = usable_df.drop_duplicates(subset=["path"])

print("Total images:", len(df))
print("Usable labelled images:", len(usable_df))
print("Unknown labels:", int((df["label"] == "Unknown").sum()))

print("\nClass distribution:")
display(usable_df["label"].value_counts().to_frame("images"))

print("\nSurface and label distribution:")
display(pd.crosstab(usable_df["surface_type"], usable_df["label"]))

usable_df.to_csv(
    PROCESSED_DIR / "usable_manifest.csv",
    index=False
)

print("\nSaved:", PROCESSED_DIR / "usable_manifest.csv")



Total images: 56092
Usable labelled images: 56092
Unknown labels: 0

Class distribution:


,images
label,
Uncracked,47608
Cracked,8484



Surface and label distribution:


label,Cracked,Uncracked
surface_type,,
Deck,2025,11595
Pavement,2608,21726
Wall,3851,14287



Saved: D:\ME AND MY LIFE\InfraScan\data\processed\usable_manifest.csv


In [2]:

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"D:\ME AND MY LIFE\InfraScan")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

usable_df = pd.read_csv(PROCESSED_DIR / "usable_manifest.csv")

# Only audit rows with a recorded content hash.
hashed = usable_df.dropna(subset=["sha256"]).copy()

duplicate_hashes = hashed[
    hashed.duplicated("sha256", keep=False)
].sort_values("sha256")

print("Rows involved in exact duplicate groups:", len(duplicate_hashes))

if len(duplicate_hashes):
    display(
        duplicate_hashes[
            ["path", "surface_type", "label", "sha256"]
        ]
    )

    summary = duplicate_hashes.groupby("sha256").agg(
        copies=("path", "size"),
        labels=("label", lambda x: ", ".join(sorted(set(x)))),
        surfaces=("surface_type", lambda x: ", ".join(sorted(set(x))))
    )

    print("\nDuplicate group summary:")
    display(summary)
else:
    print("No duplicates found in the saved manifest.")


Rows involved in exact duplicate groups: 26


,path,surface_type,label,sha256
13084,D:\ME AND MY LIFE\InfraScan\data\raw\D\UD\7053...,Deck,Uncracked,032298f2b453670ffcd9d86576418f789bc268e75f9ed8...
13083,D:\ME AND MY LIFE\InfraScan\data\raw\D\UD\7053...,Deck,Uncracked,032298f2b453670ffcd9d86576418f789bc268e75f9ed8...
1500,D:\ME AND MY LIFE\InfraScan\data\raw\D\CD\7039...,Deck,Cracked,100ed5c4918167180792a76addd82da57a57ee493ba46c...
1501,D:\ME AND MY LIFE\InfraScan\data\raw\D\CD\7039...,Deck,Cracked,100ed5c4918167180792a76addd82da57a57ee493ba46c...
1497,D:\ME AND MY LIFE\InfraScan\data\raw\D\CD\7039...,Deck,Cracked,165edba54700186384701f45fa80f160aed08f7d3f4503...
1498,D:\ME AND MY LIFE\InfraScan\data\raw\D\CD\7039...,Deck,Cracked,165edba54700186384701f45fa80f160aed08f7d3f4503...
13141,D:\ME AND MY LIFE\InfraScan\data\raw\D\UD\7053...,Deck,Uncracked,2f0a7e57a7864e541e997b7b2a3042ace6f16b045cad36...
13142,D:\ME AND MY LIFE\InfraScan\data\raw\D\UD\7053...,Deck,Uncracked,2f0a7e57a7864e541e997b7b2a3042ace6f16b045cad36...
13145,D:\ME AND MY LIFE\InfraScan\data\raw\D\UD\7053...,Deck,Uncracked,3792950659aa16111cef63e82bad69c557ac891738b14e...
13144,D:\ME AND MY LIFE\InfraScan\data\raw\D\UD\7053...,Deck,Uncracked,3792950659aa16111cef63e82bad69c557ac891738b14e...



Duplicate group summary:


,copies,labels,surfaces
sha256,,,
032298f2b453670ffcd9d86576418f789bc268e75f9ed8b7786f4d682d79eb63,2,Uncracked,Deck
100ed5c4918167180792a76addd82da57a57ee493ba46c1eb9ddc7f83ed0eadf,2,Cracked,Deck
165edba54700186384701f45fa80f160aed08f7d3f4503a4e6795ff0b1060bfa,2,Cracked,Deck
2f0a7e57a7864e541e997b7b2a3042ace6f16b045cad3621ac7123c498606f57,2,Uncracked,Deck
3792950659aa16111cef63e82bad69c557ac891738b14ec97b2a156a1d90641d,2,Uncracked,Deck
798408497ea706741a43afacf747ecfdc829ec280919e581d0fd1e4494ffd869,2,Cracked,Deck
a434807b5f462d05696c1a826c998e094764bd1f9c074d25d85cad167ee29567,2,Uncracked,Deck
b3ace75a69b5c5a28695dd9dcaddfa3b2efbf88ad533773dc1e0169a9f85520b,2,"Cracked, Uncracked",Deck
b7e4344e2015390731f17a7b6861b932f9a7c6251375eaae54fed2d1dcf0d9af,2,Uncracked,Deck


In [1]:

import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path(r"D:\ME AND MY LIFE\InfraScan")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR = PROCESSED_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED_DIR / "usable_manifest.csv")

# Identify exact-content hashes assigned to conflicting labels.
hashed = df.dropna(subset=["sha256"])
label_counts = hashed.groupby("sha256")["label"].nunique()
conflicting_hashes = set(label_counts[label_counts > 1].index)

print("Conflicting duplicate groups:", len(conflicting_hashes))

# Exclude every row belonging to a conflicting hash.
clean_df = df[~df["sha256"].isin(conflicting_hashes)].copy()

# Keep one row per exact image hash.
# Rows without hashes are retained and deduplicated by path.
has_hash = clean_df["sha256"].notna()
hashed_rows = clean_df[has_hash].drop_duplicates("sha256", keep="first")
unhashed_rows = clean_df[~has_hash].drop_duplicates("path")

clean_df = pd.concat(
    [hashed_rows, unhashed_rows],
    ignore_index=True
)

# Six-class target: surface type + crack label.
clean_df["class_name"] = (
    clean_df["surface_type"].astype(str)
    + "_"
    + clean_df["label"].astype(str)
)

print("\nFinal usable images:", len(clean_df))
print("\nClass distribution:")
display(clean_df["class_name"].value_counts().to_frame("images"))

# First split: 70% train, 30% temporary.
train_df, temp_df = train_test_split(
    clean_df,
    test_size=0.30,
    random_state=42,
    stratify=clean_df["class_name"]
)

# Split the remaining 30% equally into validation and test.
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["class_name"]
)

# Save the clean manifest and split files.
clean_df.to_csv(PROCESSED_DIR / "clean_manifest.csv", index=False)
train_df.to_csv(SPLIT_DIR / "train.csv", index=False)
val_df.to_csv(SPLIT_DIR / "validation.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test.csv", index=False)

print("\nSplit sizes:")
print(f"Train:      {len(train_df):,} ({len(train_df)/len(clean_df):.1%})")
print(f"Validation: {len(val_df):,} ({len(val_df)/len(clean_df):.1%})")
print(f"Test:       {len(test_df):,} ({len(test_df)/len(clean_df):.1%})")

print("\nSaved files:")
print("Clean manifest:", PROCESSED_DIR / "clean_manifest.csv")
print("Train:", SPLIT_DIR / "train.csv")
print("Validation:", SPLIT_DIR / "validation.csv")
print("Test:", SPLIT_DIR / "test.csv")


Conflicting duplicate groups: 2

Final usable images: 56077

Class distribution:


,images
class_name,
Pavement_Uncracked,21726
Wall_Uncracked,14286
Deck_Uncracked,11586
Wall_Cracked,3850
Pavement_Cracked,2608
Deck_Cracked,2021



Split sizes:
Train:      39,253 (70.0%)
Validation: 8,412 (15.0%)
Test:       8,412 (15.0%)

Saved files:
Clean manifest: D:\ME AND MY LIFE\InfraScan\data\processed\clean_manifest.csv
Train: D:\ME AND MY LIFE\InfraScan\data\processed\splits\train.csv
Validation: D:\ME AND MY LIFE\InfraScan\data\processed\splits\validation.csv
Test: D:\ME AND MY LIFE\InfraScan\data\processed\splits\test.csv


# Dataset Cleaning and Preparation – SDNET2018

## 1. Dataset Overview

The SDNET2018 dataset was selected for the InfraScan project, which aims to develop an AI-based system for crack detection and infrastructure damage assessment.

The dataset contains images of three concrete surface types:

- **Deck:** Concrete bridge decks
- **Pavement:** Concrete pavements
- **Wall:** Concrete walls

Each image is labelled as either **Cracked** or **Uncracked**, resulting in six classification classes.

## 2. Data Cleaning

The dataset was examined for image validity, label consistency, and duplicate images before preparing it for model training.

The following steps were performed:

1. **Image validation:** Checked image files and their metadata. All 56,092 images passed the validity checks.
2. **Label verification:** Confirmed that the images belonged to the expected surface types and crack-status categories.
3. **Duplicate detection:** Used SHA-256 hashes to identify exact duplicate images.
4. **Conflicting-label handling:** Identified two duplicate-image groups where identical image content had conflicting labels. These images were excluded to avoid introducing contradictory training examples.
5. **Duplicate removal:** Retained only one copy of each exact duplicate group with consistent labels.

After cleaning, the dataset contained **56,077 usable images**.

## 3. Final Class Distribution

| Class | Number of Images |
|---|---:|
| Pavement – Uncracked | 21,726 |
| Wall – Uncracked | 14,286 |
| Deck – Uncracked | 11,586 |
| Wall – Cracked | 3,850 |
| Pavement – Cracked | 2,608 |
| Deck – Cracked | 2,021 |
| **Total** | **56,077** |

The dataset has a noticeable class imbalance, with uncracked images outnumbering cracked images. This will be considered during model training and evaluation.

## 4. Training, Validation, and Testing Split

The cleaned dataset was divided into three subsets using a **70:15:15 split**, with stratification based on the six combined surface-type and crack-status classes.

| Dataset Split | Number of Images | Percentage |
|---|---:|---:|
| Training | 39,253 | 70% |
| Validation | 8,412 | 15% |
| Testing | 8,412 | 15% |
| **Total** | **56,077** | **100%** |

- **Training set:** Used to train the model and learn image features associated with cracks and surface types.
- **Validation set:** Used to monitor model performance during training, compare model configurations, and select the best model.
- **Testing set:** Reserved for the final evaluation of the selected model on previously unseen images.

A fixed random seed was used to make the split reproducible.

## 5. Output Files

The cleaned dataset information and split assignments were saved as CSV files:

- `data/processed/clean_manifest.csv` – Contains the cleaned image manifest.
- `data/processed/splits/train.csv` – Contains training image paths and labels.
- `data/processed/splits/validation.csv` – Contains validation image paths and labels.
- `data/processed/splits/test.csv` – Contains testing image paths and labels.

The original dataset files were not modified.

## 6. Next Step

The next stage is to develop a baseline deep-learning classification model using transfer learning with ResNet-18. Class-weighted loss and metrics such as precision, recall, macro-F1 score, and the confusion matrix will be considered to address and evaluate performance under class imbalance.

This baseline will provide a foundation for extending InfraScan toward more advanced features, including crack localization, segmentation, severity assessment, and explainable AI.

**Important limitation:** The current classification task predicts the surface type and whether a crack is present. It does not, by itself, estimate crack dimensions, structural safety, or the remaining service life of a building. Those objectives require additional methods and appropriate validation.